# TOKATCI — lean4 context-tree training (Kaggle)

Input: **`tokatci_tree_data.zip`**
- `dataset.npz`: lean4 family action tree dataset
- `quality_labels.csv`: good/neutral/bad labels from 146 fresh noAI games

This notebook trains **phase + context** action trees:
- no matchup-specific trees
- small sklearn decision trees (no deep learning)
- episode-grouped validation
- top-1 and top-3 recall
- optional state-value feature and quality-label sample weights

No replays/engine are needed. No submission is made.


In [ ]:
from pathlib import Path
import json, csv, zipfile, shutil
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import HistGradientBoostingClassifier

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
WORK.mkdir(exist_ok=True)

# Find or extract the data zip.
z = next(iter(INPUT.rglob('tokatci_tree_data.zip')), None)
if z is not None:
    dest = WORK/'tokatci_tree_data'
    if dest.exists(): shutil.rmtree(dest)
    dest.mkdir()
    with zipfile.ZipFile(z) as f: f.extractall(dest)
    DATA = next(dest.rglob('dataset.npz'))
    QUAL = next(dest.rglob('quality_labels.csv'))
else:
    DATA = next(iter(INPUT.rglob('dataset.npz')), None)
    QUAL = next(iter(INPUT.rglob('quality_labels.csv')), None)
if DATA is None or QUAL is None:
    raise FileNotFoundError('dataset.npz / quality_labels.csv bulunamadi')
print('DATA', DATA)
print('QUAL', QUAL)
d = np.load(DATA)
print('rows', len(d['label']), 'episodes', len(np.unique(d['episode_id'])))


In [ ]:
# Feature builder: state + option + context + opponent + sequence + optional value.
OPP = ['Dragapult','Mega Kangaskhan','Mega Excadrill','Mega Lopunny','Banette',
       'Alakazam','Hydrapple','Mega Abomasnow','Festival Lead','Mega Lucario',
       'Archaludon','Slowking-Kangaskhan','Teal Mask Ogerpon',"Marnie's Grimmsnarl"]

def build_X(d, value=None):
    n = len(d['label'])
    opp = np.zeros((n, len(OPP)+1), np.float32)
    opp[np.arange(n), d['opp_code']] = 1
    last = np.zeros((n, 18), np.float32); last[np.arange(n), d['last_type']] = 1
    prev = np.zeros((n, 18), np.float32); prev[np.arange(n), d['prev_type']] = 1
    parts = [
        d['state'].astype(np.float32),
        d['option'].astype(np.float32),
        d['context'][:,None].astype(np.float32),
        d['select_type'][:,None].astype(np.float32),
        d['min_count'][:,None].astype(np.float32),
        d['max_count'][:,None].astype(np.float32),
        d['turn'][:,None].astype(np.float32) / 20.0,
        opp, last, prev,
    ]
    if value is not None:
        parts.append(value[:,None].astype(np.float32))
    return np.concatenate(parts, axis=1)


In [ ]:
# Episode split, base weights, quality map, and value model.
ep = d['episode_id']; step = d['step_index']
u = np.unique(ep); rng = np.random.default_rng(47); rng.shuffle(u)
nv = max(1, int(len(u) * 0.2)); val = set(u[:nv].tolist())
tr = np.array([int(e) not in val for e in ep]); va = ~tr
base = np.where(d['won'] > 0, 2.0, 0.5) * (1.0 + 0.05*np.abs(d['prize_diff']))
q = {}
for r in csv.DictReader(open(QUAL, encoding='utf-8')):
    q[(int(r['episode_id']), int(r['step']))] = r['label']
vm = HistGradientBoostingClassifier(max_iter=120, learning_rate=.06, max_leaf_nodes=15,
                                    l2_regularization=.1, random_state=47)
vm.fit(d['state'][tr].astype(np.float32), d['won'][tr])
value = vm.predict_proba(d['state'].astype(np.float32))[:,1].astype(np.float32)
print('split', int(tr.sum()), int(va.sum()), 'quality labels', len(q))


In [ ]:
from collections import defaultdict

def recall(pred, va, topk=1):
    y = d['label']; groups = defaultdict(list)
    for i in np.nonzero(va)[0]:
        groups[(int(ep[i]), int(step[i]))].append((float(pred[i]), int(i)))
    c = t = 0
    for a in groups.values():
        a.sort(reverse=True)
        nsel = int(y[[i for _,i in a]].sum())
        k = max(1, min(topk, nsel if topk == 1 else len(a)))
        c += len({i for _,i in a[:k]} & {i for _,i in a if y[i] == 1})
        t += len({i for _,i in a if y[i] == 1})
    return c / max(1, t)

def run_config(depth=16, leaf=120, use_value=True, quality=True, good_factor=2.0, bad_factor=0.2):
    X = build_X(d, value if use_value else None)
    w = base.copy()
    nq = 0
    if quality:
        for i in range(len(w)):
            if d['label'][i] != 1: continue
            lab = q.get((int(ep[i]), int(step[i])))
            if lab == 'good': w[i] *= good_factor; nq += 1
            elif lab == 'bad': w[i] *= bad_factor; nq += 1
    ph = d['phase']; ctx = d['context']
    pred = np.full(len(ep), -1.0, np.float32); nctx = 0
    # FAST: phase fallback first, then only the largest contexts.
    CONTEXTS = [0, 7, 14, 8, 3, 4, 21, 5]
    MAX_ROWS = 120_000
    rng_fit = np.random.default_rng(123)
    for phase in [0, 1, 2]:
        mtr = tr & (ph == phase); mva = va & (ph == phase)
        if mtr.sum() >= 400 and mva.sum() >= 50:
            idx = np.nonzero(mtr)[0]
            if len(idx) > MAX_ROWS:
                idx = rng_fit.choice(idx, MAX_ROWS, replace=False)
            clf = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf,
                                         class_weight='balanced', random_state=47)
            clf.fit(X[idx], d['label'][idx], sample_weight=w[idx])
            pred[np.nonzero(mva)[0]] = clf.predict_proba(X[mva])[:,1]
        for cx in CONTEXTS:
            mtr = tr & (ph == phase) & (ctx == cx)
            mva = va & (ph == phase) & (ctx == cx)
            if mtr.sum() < 400 or mva.sum() < 50: continue
            idx = np.nonzero(mtr)[0]
            if len(idx) > MAX_ROWS:
                idx = rng_fit.choice(idx, MAX_ROWS, replace=False)
            clf = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf,
                                         class_weight='balanced', random_state=47)
            clf.fit(X[idx], d['label'][idx], sample_weight=w[idx])
            pred[np.nonzero(mva)[0]] = clf.predict_proba(X[mva])[:,1]
            nctx += 1
    return {
        'depth': depth, 'leaf': leaf, 'use_value': use_value, 'quality': quality,
        'good_factor': good_factor, 'bad_factor': bad_factor,
        'contexts': nctx, 'top1': recall(pred, va, 1), 'top3': recall(pred, va, 3),
        'labeled_selected': nq,
    }


In [ ]:
# Focused/fast grid: best known region.
# Full grid 36 configs is slow because each config trains 35 context trees.
FAST_MODE = True
if FAST_MODE:
    # one strong config; phase fallback + top contexts + row cap
    configs = [(16, 120, True, True)]
else:
    configs = [(d,l,v,q) for d in [14,16,18] for l in [80,120,160] for v in [False,True] for q in [False,True]]

runs = []
import time
for depth, leaf, use_value, quality in configs:
    t0 = time.time()
    r = run_config(depth=depth, leaf=leaf, use_value=use_value, quality=quality)
    r["seconds"] = round(time.time()-t0, 1)
    runs.append(r)
    print(r, flush=True)
    # Persist partial results every config; safe for overnight runs.
    (WORK/'runs.json').write_text(json.dumps(runs, indent=2))
    (WORK/'best_partial.json').write_text(json.dumps(max(runs, key=lambda x: (x['top1'], x['top3'])), indent=2))
best = max(runs, key=lambda r: (r['top1'], r['top3']))
print('BEST', best)
(WORK/'runs.json').write_text(json.dumps(runs, indent=2))
(WORK/'best.json').write_text(json.dumps(best, indent=2))


In [ ]:
# Optional: export one representative phase/context tree text for inspection.
# Uses the best config; main phase=1, context=0.
best_depth = best['depth']; best_leaf = best['leaf']
X = build_X(d, value if best['use_value'] else None)
w = base.copy()
if best['quality']:
    for i in range(len(w)):
        if d['label'][i] != 1: continue
        lab = q.get((int(ep[i]), int(step[i])))
        if lab == 'good': w[i] *= best['good_factor']
        elif lab == 'bad': w[i] *= best['bad_factor']
m = tr & (d['phase'] == 1) & (d['context'] == 0)
clf = DecisionTreeClassifier(max_depth=best_depth, min_samples_leaf=best_leaf,
                             class_weight='balanced', random_state=47)
clf.fit(X[m], d['label'][m], sample_weight=w[m])
txt = export_text(clf, max_depth=5, decimals=2)
(WORK/'best_phase1_main.txt').write_text(txt)
print(txt[:8000])
print('wrote', WORK/'best.json', WORK/'runs.json')
